In [19]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Preprocessing environment ready.")

Preprocessing environment ready.


In [20]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = (
    PROJECT_ROOT
    / "dataset"
    / "raw"
    / "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)

PROCESSED_DATA = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
)

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw dataset:")
print(RAW_DATA)

print("\nProcessed dataset:")
print(PROCESSED_DATA)

Project root:
d:\BrainTumorHybrid

Raw dataset:
d:\BrainTumorHybrid\dataset\raw\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData

Processed dataset:
d:\BrainTumorHybrid\dataset\processed


In [21]:
import re

# Collect all NIfTI files
nifti_files = list(RAW_DATA.rglob("*.nii.gz"))

print("Total NIfTI files:", len(nifti_files))


# Group files by actual patient ID
patients = {}

for file in nifti_files:

    match = re.match(
        r"^(BraTS-GLI-\d+)-\d+-",
        file.name
    )

    if match is None:
        print("Could not identify patient:", file.name)
        continue

    patient_id = match.group(1)

    if patient_id not in patients:
        patients[patient_id] = []

    patients[patient_id].append(file)


print("Total patients:", len(patients))

Total NIfTI files: 6255
Total patients: 1133


In [22]:
required_modalities = [
    "t1n",
    "t1c",
    "t2w",
    "t2f",
    "seg"
]

def get_modality_file(files, modality):
    matches = [
        f for f in files
        if f.name.lower().endswith(
            f"-{modality}.nii.gz"
        )
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {modality} in files:\n"
            + "\n".join(f.name for f in files)
        )

    return matches[0]
complete_patients = []

for patient_id, files in patients.items():

    modality_files = {
        modality: get_modality_file(files, modality)
        for modality in required_modalities
    }

    if all(path is not None for path in modality_files.values()):
        complete_patients.append(patient_id)


print("Complete patients:", len(complete_patients))
print("Incomplete patients:", len(patients) - len(complete_patients))

Complete patients: 1133
Incomplete patients: 0


In [23]:
incomplete_details = []

for patient_id, files in patients.items():

    found = set()

    for file in files:
        name = file.name.lower()

        for modality in required_modalities:
            if name.endswith(f"-{modality}.nii.gz"):
                found.add(modality)

    missing = set(required_modalities) - found

    if missing:
        incomplete_details.append(
            {
                "patient_id": patient_id,
                "files": len(files),
                "missing": sorted(missing)
            }
        )

print("Incomplete patients:", len(incomplete_details))

for item in incomplete_details[:20]:
    print(
        item["patient_id"],
        "| files:", item["files"],
        "| missing:", item["missing"]
    )

Incomplete patients: 0


In [24]:
complete_patients = sorted(patients.keys())

print("Usable patients:", len(complete_patients))

Usable patients: 1133


In [25]:
import random

SEED = 42

rng = random.Random(SEED)

selected_patients = sorted(
    rng.sample(complete_patients, 20)
)

print("Selected patients:", len(selected_patients))

for patient_id in selected_patients:
    print(patient_id)

Selected patients: 20
BraTS-GLI-00077
BraTS-GLI-00084
BraTS-GLI-00095
BraTS-GLI-00099
BraTS-GLI-00251
BraTS-GLI-00271
BraTS-GLI-00296
BraTS-GLI-00320
BraTS-GLI-00403
BraTS-GLI-00579
BraTS-GLI-00628
BraTS-GLI-00638
BraTS-GLI-00646
BraTS-GLI-00679
BraTS-GLI-00715
BraTS-GLI-00807
BraTS-GLI-01275
BraTS-GLI-01280
BraTS-GLI-01450
BraTS-GLI-01532


In [26]:
rng = random.Random(SEED)

shuffled_patients = selected_patients.copy()
rng.shuffle(shuffled_patients)

train_patients = shuffled_patients[:14]
val_patients = shuffled_patients[14:17]
test_patients = shuffled_patients[17:20]

print("TRAIN:", len(train_patients))
print("VALIDATION:", len(val_patients))
print("TEST:", len(test_patients))

TRAIN: 14
VALIDATION: 3
TEST: 3


In [27]:
MAX_TUMOR_SLICES = 20
BACKGROUND_RATIO = 1.0

metadata_rows = []


def create_slice_records(patient_id, split):
    files = patients[patient_id]

    # Locate the five required files
    modality_files = {
        modality: get_modality_file(files, modality)
        for modality in required_modalities
    }

    # Load segmentation mask
    seg = nib.load(
        modality_files["seg"]
    ).get_fdata()

    # Find slices containing tumor
    tumor_slices = np.where(
        np.any(seg > 0, axis=(0, 1))
    )[0]

    # Find slices without tumor
    all_slices = np.arange(seg.shape[2])

    tumor_set = set(tumor_slices)

    background_slices = np.array([
        i for i in all_slices
        if i not in tumor_set
    ])

    # Reproducible random selection
    patient_rng = np.random.default_rng(
        SEED + int(patient_id[-5:])
    )

    # Limit tumor-containing slices
    if len(tumor_slices) > MAX_TUMOR_SLICES:
        tumor_slices = patient_rng.choice(
            tumor_slices,
            size=MAX_TUMOR_SLICES,
            replace=False
        )

    # Select equal number of background slices
    num_background = min(
        len(background_slices),
        int(len(tumor_slices) * BACKGROUND_RATIO)
    )

    if num_background > 0:
        background_slices = patient_rng.choice(
            background_slices,
            size=num_background,
            replace=False
        )

    # Add tumor slices
    for slice_idx in tumor_slices:
        metadata_rows.append({
            "patient_id": patient_id,
            "split": split,
            "slice_index": int(slice_idx),
            "has_tumor": 1
        })

    # Add background slices
    for slice_idx in background_slices:
        metadata_rows.append({
            "patient_id": patient_id,
            "split": split,
            "slice_index": int(slice_idx),
            "has_tumor": 0
        })


# Build metadata
for patient_id in train_patients:
    create_slice_records(patient_id, "train")

for patient_id in val_patients:
    create_slice_records(patient_id, "val")

for patient_id in test_patients:
    create_slice_records(patient_id, "test")


metadata = pd.DataFrame(metadata_rows)

print("Total 2D samples:", len(metadata))

print("\nSamples by split:")
print(metadata["split"].value_counts())

print("\nTumor/background:")
print(metadata["has_tumor"].value_counts())

Total 2D samples: 800

Samples by split:
split
train    560
val      120
test     120
Name: count, dtype: int64

Tumor/background:
has_tumor
1    400
0    400
Name: count, dtype: int64


In [28]:
print(metadata.head(10))
print()
print(metadata.tail(10))

        patient_id  split  slice_index  has_tumor
0  BraTS-GLI-01532  train          105          1
1  BraTS-GLI-01532  train           66          1
2  BraTS-GLI-01532  train          124          1
3  BraTS-GLI-01532  train          115          1
4  BraTS-GLI-01532  train           82          1
5  BraTS-GLI-01532  train          128          1
6  BraTS-GLI-01532  train          102          1
7  BraTS-GLI-01532  train          106          1
8  BraTS-GLI-01532  train           75          1
9  BraTS-GLI-01532  train           68          1

          patient_id split  slice_index  has_tumor
790  BraTS-GLI-00099  test            8          0
791  BraTS-GLI-00099  test           12          0
792  BraTS-GLI-00099  test            1          0
793  BraTS-GLI-00099  test          154          0
794  BraTS-GLI-00099  test           21          0
795  BraTS-GLI-00099  test           10          0
796  BraTS-GLI-00099  test            7          0
797  BraTS-GLI-00099  test           15  

In [29]:
print("\nSamples per patient:")
print(
    metadata.groupby(["split", "patient_id"])
    .size()
)


Samples per patient:
split  patient_id     
test   BraTS-GLI-00077    40
       BraTS-GLI-00099    40
       BraTS-GLI-00403    40
train  BraTS-GLI-00084    40
       BraTS-GLI-00251    40
       BraTS-GLI-00271    40
       BraTS-GLI-00296    40
       BraTS-GLI-00579    40
       BraTS-GLI-00628    40
       BraTS-GLI-00638    40
       BraTS-GLI-00646    40
       BraTS-GLI-00679    40
       BraTS-GLI-00715    40
       BraTS-GLI-00807    40
       BraTS-GLI-01280    40
       BraTS-GLI-01450    40
       BraTS-GLI-01532    40
val    BraTS-GLI-00095    40
       BraTS-GLI-00320    40
       BraTS-GLI-01275    40
dtype: int64


In [30]:
metadata_path = PROCESSED_DATA / "slice_metadata.csv"

metadata.to_csv(
    metadata_path,
    index=False
)

print("Saved metadata to:")
print(metadata_path)

Saved metadata to:
d:\BrainTumorHybrid\dataset\processed\slice_metadata.csv
